In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, lit, to_date, max as spark_max, row_number, coalesce
from pyspark.sql.window import Window
from lib.aws_s3 import warehouse_dir

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("17 - Service Production")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "service"
schema_name = "prd"
table_name = f"fact_{table}"
table_full_name = f"{schema_name}.{table_name}"
staging_table_full_name = f"stg.fact_{table}_stg"

print("****************************************")
print("17 - Service Production")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print(f"staging_table_full_name = {staging_table_full_name}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Read data from staging
df_stg = (
    spark
    .table(staging_table_full_name)
).fillna('N/A', subset=['operation_code', 'part_code'])

In [ ]:
print(f"df_stg: {df_stg.count()} records read from {staging_table_full_name}")
df_stg.show()
df_stg.printSchema()

In [ ]:
# Identify customers in the service file that are not current in dim_customer
df_missing_cust = (
    df_stg.select("customer_id").distinct()
          .join(spark.read.table("prd.dim_customer").where("is_current").selectExpr("customer_id"), "customer_id", "left_anti")
)

# If any, insert inferred records to prd.dim_customer
if df_missing_cust.count() > 0:
    print("Early arriving facts identified!")
    print(f"{df_missing_cust.count()} new customer(s) identified.")
    print("Inserting inferred records to prd.dim_customer table!")
    
    # Find current maximum customer_key value
    max_key = (
        spark.read.table("prd.dim_customer")
        .where("customer_key > 0")
        .agg(coalesce(spark_max("customer_key"), lit(0)).alias("max_key"))
        .first()["max_key"]
    )
    print(f"Current maximum customer_key value is {max_key}")

    w = Window.orderBy("customer_id")

    # Create placeholder records
    df_inferred = (
        df_missing_cust
        .withColumn("customer_key", (row_number().over(w) + lit(max_key)).cast("long"))
        .withColumn("first_name",   lit(None).cast("string"))
        .withColumn("last_name",    lit(None).cast("string"))
        .withColumn("address",      lit(None).cast("string"))
        .withColumn("city",         lit(None).cast("string"))
        .withColumn("state",        lit(None).cast("string"))
        .withColumn("zipcode",      lit(None).cast("string"))
        .withColumn("email",        lit(None).cast("string"))
        .withColumn("phone_number", lit(None).cast("string"))
        .withColumn("customer_type", lit(None).cast("string"))
        .withColumn("is_current", lit(True))
        .withColumn("effective_start_date", to_date(lit(run_date), "yyyyMMdd"))
        .withColumn("effective_end_date",   to_date(lit("9999-12-31")))
        .withColumn("is_inferred", lit(True))
        .withColumn("insert_ts", lit(now_ts))
        .withColumn("update_ts", lit(now_ts))
    )

    # show new records to insert
    print("Following records will be inferred:")
    df_inferred.show()

    # Appending new customers to prd.dim_customer
    (
        df_inferred.write
        .format("delta")
        .mode("append")
        .saveAsTable("prd.dim_customer")
    )

    # Displaying new records
    print("Inferred records from prd.dim_customer:")
    spark.sql(f"select * from prd.dim_customer where customer_key > {max_key}").show()

In [ ]:
# Read dimension tables to get dimension keys

# dealer
df_dim_dealer = spark.read.table("prd.dim_dealer").where("is_current").selectExpr("dealer_id", "dealer_key")

# customer
df_dim_customer = spark.read.table("prd.dim_customer").where("is_current").selectExpr("customer_id", "customer_key")

# operation
df_dim_operation = spark.read.table("prd.dim_operation").selectExpr("operation_code", "operation_key")

# part
df_dim_part = spark.read.table("prd.dim_part").selectExpr("part_code", "part_key")

In [ ]:
df_service = (
    df_stg
    .join(df_dim_dealer, how = 'left', on=df_dim_dealer.dealer_id == df_stg.dealer_id)
    .join(df_dim_customer, how = 'left', on=df_dim_customer.customer_id == df_stg.customer_id)
    .join(df_dim_operation, how = 'left', on=df_dim_operation.operation_code == df_stg.operation_code)
    .join(df_dim_part, how = 'left', on=df_dim_part.part_code == df_stg.part_code)
    .withColumn("insert_ts", lit(now_ts))
    .selectExpr(
        'ro_number',
        'line_number',
        'dealer_key',
        'customer_key',
        'vin',
        'cast(open_date as date) as service_open_date',
        'cast(close_date as date) as service_close_date',
        'odometer',
        'operation_key',
        'part_key',
        'line_type',
        'pay_type',
        'quantity',
        'unit_price',
        'line_amount',
        'insert_ts'
    )
)

In [ ]:
print("Added audit columns")
print(f"df_service: {df_service.count()} records")
df_service.show()
df_service.printSchema()

In [ ]:
# Load data to prd schema

(
    df_service.write
    .format("delta")
    .mode("append")
    .saveAsTable(table_full_name)
)

print(f"{df_service.count()} records written to {table_full_name} (mode=append)")

In [ ]:
print(f"Data loaded in this run:")
spark.table(f'{schema_name}.fact_{table}').where(f"insert_ts = '{now_ts}'").show()

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# Stop spark session
#spark.stop()